# Week 2 practical: Data preprocessing, Part B
**CHS2406: Data Driven AI · Dr Hamid Nasiri · University of Huddersfield**

**Student workbook · 60 minutes · Google Colab · CPU only**

Learn how to inspect relationships, flag unusual observations, scale and encode features,
and build a pipeline that evaluates imbalanced classification fairly.

The examples follow the attached Week 2, Lecture 2 slides. All data are generated here;
no downloads, accounts, API keys or uploads of datasets are needed.

## Session route

| Minutes | Activity | Slide links |
|:---|:---|:---|
| 0–5 | Setup and load the teaching data | 62 |
| 5–15 | Correlation, z-scores and box plots; Exercise 1 | 6–15, 34–42 |
| 15–25 | Compare four scalers; Exercise 2 | 16–27 |
| 25–40 | Encoding and a complete preprocessing pipeline; Exercise 3 | 29–33, 43–47 |
| 40–55 | Stratification, imbalance and SMOTE; Exercise 4 | 48–60, 63–67 |
| 55–60 | Exit questions and save your work | Review |

**Finish Sections 0–5 during the practical.** Extensions A–D are optional work for later.
The stated times include reading, running examples, coding and discussion.

By the end, you should be able to fit transformations on training data, encode nominal
and ordinal features appropriately, explain why an outlier needs investigation, and
compare minority-class precision and recall before and after resampling.

### How to use this notebook in Colab

1. Open [Google Colab](https://colab.research.google.com/), choose **File → Upload notebook**
   and select this `.ipynb` file. Use a Python 3 runtime; a GPU is unnecessary.
2. Run cells from top to bottom with **Shift+Enter**, starting with installation.
3. In each exercise, replace the marked `None` values. Starter cells print a reminder
   until you complete them, so unfinished exercises do not stop later worked examples.
4. Enter short explanations in the answer cells. Save your copy before leaving.

If installation asks you to restart the session, restart and run from the top again.
If you see a package compatibility error, restart after running the installation cell.

## 0. Setup and teaching data · 5 minutes

Run the next two cells. The installation command includes the required libraries and pins
scikit-learn and imbalanced-learn to the versions used to check this practical.
The package is installed as **`imbalanced-learn`** and imported as **`imblearn`**.
NumPy and SciPy are included explicitly so the notebook also works in a fresh Python environment.

In [ ]:
%pip -q install "numpy>=1.25.2,<3" "scipy>=1.11.4,<2" "pandas>=2.2,<3" "matplotlib>=3.8,<4" "seaborn>=0.13,<0.14" "scikit-learn==1.8.0" "imbalanced-learn==0.14.2"

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from importlib.metadata import version

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.datasets import make_classification
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, ConfusionMatrixDisplay,
)
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, MaxAbsScaler, RobustScaler,
    OneHotEncoder, OrdinalEncoder,
)
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

SEED = 42
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False,
                     "axes.spines.right": False})
pd.set_option("display.max_columns", 12)
for package in ["numpy", "pandas", "scikit-learn", "imbalanced-learn"]:
    print(f"{package}: {version(package)}")

### A synthetic course dataset

Each row represents an invented student. We will predict `passed` (1 = pass, 0 = did not pass)
using four numeric features, a nominal feature (`study_mode`) and an ordinal feature
(`experience`). The generator deliberately adds a few missing numeric values.
These invented relationships are for coding practice and are not findings about real students.

Run the supplied cell. We split **before** inspecting relationships or fitting transformations.
`stratify=y` keeps approximately the same pass rate in both partitions.

In [ ]:
# Supplied setup: you do not need to edit this generator.
rng = np.random.default_rng(SEED)
n = 240
study = rng.uniform(1, 15, n)
attendance = rng.uniform(40, 100, n)
experience = rng.choice(["Beginner", "Intermediate", "Advanced"], n,
                        p=[0.45, 0.35, 0.20])
experience_effect = pd.Series(experience).map(
    {"Beginner": 0, "Intermediate": 4, "Advanced": 8}
).to_numpy()
score = (25 + 2.5 * study + 0.28 * (attendance - 40)
         + experience_effect + rng.normal(0, 8, n))
course = pd.DataFrame({
    "study_hours": study.round(1),
    "attendance_pct": attendance.round(1),
    "practice_questions": np.maximum(0, 3 * study + rng.normal(0, 4, n)).round(),
    "commute_minutes": rng.integers(0, 61, n),
    "study_mode": rng.choice(["Campus", "Hybrid", "Online"], n),
    "experience": experience,
    "passed": (score >= 60).astype(int),
})
for column in ["study_hours", "attendance_pct"]:
    course.loc[rng.choice(n, 10, replace=False), column] = np.nan

X = course.drop(columns="passed")  # Keep the target out of the inputs.
y = course["passed"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED
)
numeric_columns = ["study_hours", "attendance_pct", "practice_questions", "commute_minutes"]
display(X_train.head())
print("Train/test shapes:", X_train.shape, X_test.shape)
print("Training missing values:", X_train.isna().sum().to_dict())
print("Train/test pass rates:", round(y_train.mean(), 3), round(y_test.mean(), 3))

## 1. Correlation and outliers · 10 minutes

### Worked example: a correlation heatmap

Pearson correlation describes a **linear** relationship. Its values lie between −1 and +1.
A value near zero does not rule out a nonlinear relationship or a useful combination of features.
Correlation also does not establish causation.

The target is included in this diagnostic heatmap for **training rows only**. It remains
excluded from `X_train`. With a 0/1 target, Pearson correlation is also called point-biserial correlation.
Pandas uses available pairs when a column contains missing values.

In [ ]:
training_view = X_train[numeric_columns].assign(passed=y_train)
corr = training_view.corr(method="pearson")
fig, ax = plt.subplots(figsize=(7.5, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", vmin=-1, vmax=1,
            center=0, square=True, ax=ax, cbar_kws={"label": "Pearson r"})
ax.set_title("Training data: numeric features and target")
plt.tight_layout()
plt.show()

# Covariance depends on units; correlation is unitless.
pair = X_train[["study_hours", "attendance_pct"]]
print("Sample covariance:")
display(pair.cov().round(2))
print("Which input has the largest absolute correlation with passed?")

**Discuss:** Which two inputs may contain similar information? Why should we avoid
concluding that an input is useless just because its Pearson correlation is small?

### Worked example: outlier rules

The lecture's completion-time example has 25 observations: 8 minutes occurs 4 times,
9 occurs 8 times, 10 occurs 4 times, 11 occurs 8 times, and 18 occurs once.

For this descriptive example, use the sample standard deviation (`ddof=1`):

$$z_i = \frac{x_i-\bar{x}}{s},\qquad
s=\sqrt{\frac{\sum_i(x_i-\bar{x})^2}{n-1}}.$$

A common rule flags $|z_i|>3$. A box plot uses the fences
$Q_1-1.5\,IQR$ and $Q_3+1.5\,IQR$, where $IQR=Q_3-Q_1$.
The whiskers reach the most extreme **observations within** the fences.
The rules flag values for investigation; they do not prove a data error.

In [ ]:
times = pd.Series([8]*4 + [9]*8 + [10]*4 + [11]*8 + [18], name="minutes")
mean_time = times.mean()
sample_sd = times.std(ddof=1)
q1, q3 = times.quantile([0.25, 0.75])
iqr = q3 - q1
lower_fence, upper_fence = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"Mean={mean_time:.1f}, sample SD={sample_sd:.1f}")
print(f"Q1={q1:.1f}, Q3={q3:.1f}, IQR={iqr:.1f}")
print(f"Fences: [{lower_fence:.1f}, {upper_fence:.1f}]")
fig, ax = plt.subplots(figsize=(7, 2.5))
sns.boxplot(x=times, whis=1.5, color="#8ac6c3", ax=ax)
ax.axvline(lower_fence, color="#b86230", linestyle="--", label="IQR fences")
ax.axvline(upper_fence, color="#b86230", linestyle="--")
ax.set(xlabel="Completion time (minutes)", title="Flags identify observations to investigate")
ax.legend(loc="upper center")
plt.tight_layout()
plt.show()

### Exercise 1 · Flag unusual completion times


Replace the three `None` values to compute each z-score, create the Boolean condition
`abs(z) > 3`, and find observations outside the IQR fences. Use `|` for elementwise OR
and put each comparison in parentheses.

**Explain:** Do the two methods flag the same observation? Give one reason to keep it
and one reason to correct or remove it. No rows are removed in this practical.

In [ ]:
z_scores = None       # TODO: subtract mean_time, then divide by sample_sd.
z_flags = None        # TODO: Boolean Series for absolute z-score > 3.
iqr_flags = None      # TODO: below lower_fence OR above upper_fence.

if z_scores is None or z_flags is None or iqr_flags is None:
    print("Complete the three TODOs, then rerun this cell.")
else:
    result = pd.DataFrame({"minutes": times, "z_score": z_scores,
                           "z_flag": z_flags, "IQR_flag": iqr_flags})
    display(result[result["z_flag"] | result["IQR_flag"]])
    assert result["z_flag"].dtype == bool, "Use a Boolean comparison."
    assert result.loc[times.idxmax(), "z_flag"], "Check the z-score formula."

**Your explanation:** Replace this sentence with your answer.

## 2. Feature scaling · 10 minutes

The lecture compares four transformations. Each works independently on each numeric feature.
The statistics below come from the training observations.

| Transformer | Transformation | What to remember |
|:---|:---|:---|
| `StandardScaler` | `(x − mean) / SD` | Training mean 0 and population SD 1 for nonconstant features; sensitive to extremes |
| `MinMaxScaler` | `(x − min) / (max − min)` | Training range [0, 1]; later values can fall outside it |
| `MaxAbsScaler` | `x / max(abs(x))` | Preserves zeros; useful with sparse data |
| `RobustScaler` | `(x − median) / IQR` | Training median 0 and IQR 1; extreme values remain |

`StandardScaler` uses population SD (`ddof=0`). It neither requires nor creates a normal
 distribution. `RobustScaler` does not generally put Q1 at −1 or Q3 at +1.

In [ ]:
# Standalone teaching data, matching the robust-scaling example in the slides.
commute_train = np.array([10, 12, 14, 16, 18, 20, 22, 24, 100], dtype=float).reshape(-1, 1)
scalers = {"Standard": StandardScaler(), "Min–max": MinMaxScaler(),
           "MaxAbs": MaxAbsScaler(), "Robust": RobustScaler()}
scaled_values = {name: scaler.fit_transform(commute_train).ravel()
                 for name, scaler in scalers.items()}
comparison = pd.DataFrame({"Original": commute_train.ravel(), **scaled_values})
display(comparison.round(3))

fig, axes = plt.subplots(1, 4, figsize=(12, 3.2))
for ax, (name, values) in zip(axes, scaled_values.items()):
    ax.scatter(np.arange(1, 9), values[:8], color="#176c9c", label="Typical values")
    ax.scatter(9, values[8], color="#d05d32", marker="D", label="100-minute value")
    ax.set(title=name, xlabel="Observation", ylabel="Transformed value")
fig.suptitle("Same observations, four scalers (each panel has its own y-axis)")
plt.tight_layout()
plt.show()

robust_values = scaled_values["Robust"]
print("Robust median:", np.median(robust_values))
print("Robust Q1/Q3:", np.quantile(robust_values, [0.25, 0.75]))

**Predict before running Exercise 2:** A min–max scaler learns from `[10, 20, 30]`.
Where should it place a new value of `40`?

For predictive modelling: split first, use `fit_transform` on training inputs, then
`transform` on held-out inputs with the **same fitted object**. The classification target
stays as its 0/1 labels.

### Exercise 2 · Reuse a fitted scaler


Create a `MinMaxScaler`. Transform the training values with `fit_transform`, then transform
the held-out values with `transform`. Use the same scaler object for both.

**Explain:** Why can the held-out result exceed 1? What would go wrong if you fitted the
scaler again on the held-out values?

In [ ]:
tiny_train = np.array([[10.0], [20.0], [30.0]])
tiny_test = np.array([[40.0]])
my_scaler = None        # TODO: construct the scaler.

if my_scaler is None:
    print("Create my_scaler and complete both transformations below.")
else:
    train_scaled = None    # TODO: learn from and transform tiny_train.
    test_scaled = None     # TODO: only transform tiny_test.
    if train_scaled is None or test_scaled is None:
        print("Complete both transformations.")
    else:
        print("Training values:", train_scaled.ravel())
        print("Held-out values:", test_scaled.ravel())
        assert np.allclose(my_scaler.data_min_, tiny_train.min(axis=0)), "Do not refit on test data."
        assert np.allclose(my_scaler.data_max_, tiny_train.max(axis=0)), "Keep training statistics."
        assert test_scaled.shape == tiny_test.shape

**Your explanation:** Replace this sentence with your answer.

## 3. Encoding and pipelines · 15 minutes

### Worked example: nominal and ordinal features

`study_mode` has no natural ranking, so use one-hot encoding. `experience` has an explicit
order, so use ordinal encoding with that order supplied. Ordinal codes express order;
a linear model also treats differences between consecutive codes as equal, which is an
extra modelling assumption.

`handle_unknown="ignore"` makes an unseen nominal category produce zeros for that
feature's one-hot columns. This prevents an error but gives the model no learned effect
for the new category. The fitted training categories still define the output columns.

In [ ]:
mode_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
mode_encoder.fit(X_train[["study_mode"]])
new_modes = pd.DataFrame({"study_mode": ["Campus", "Online", "Evening"]})
display(pd.DataFrame(mode_encoder.transform(new_modes),
                     columns=mode_encoder.get_feature_names_out(),
                     index=new_modes["study_mode"]))

experience_order = [["Beginner", "Intermediate", "Advanced"]]
experience_encoder = OrdinalEncoder(categories=experience_order)
experience_encoder.fit(X_train[["experience"]])
ordered_example = pd.DataFrame({"experience": ["Advanced", "Beginner", "Intermediate"]})
display(ordered_example.assign(code=experience_encoder.transform(ordered_example).ravel()))

### Worked example: one object for preprocessing and prediction

`ColumnTransformer` sends different columns to different preprocessing branches. Here:

- Numeric columns: fill missing values with the training median, then standardise.
- Study mode: fill missing values with the training mode, then one-hot encode.
- Experience: fill missing values with the training mode, then apply the explicit order.

The outer `Pipeline` applies these branches and fits logistic regression. Calling
`predict` repeats the fitted transformations automatically. It does not refit them.
The classifier is just a vehicle for practising preprocessing; no parameter tuning is required.

In [ ]:
numeric_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
nominal_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
ordinal_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("encode", OrdinalEncoder(categories=experience_order)),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipe, numeric_columns),
    ("mode", nominal_pipe, ["study_mode"]),
    ("experience", ordinal_pipe, ["experience"]),
])
course_model = Pipeline([
    ("preprocess", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000)),
])
print("Pipeline ready. It has not seen any training or test rows yet.")

### Exercise 3 · Fit and predict using raw tables


Fit `course_model` using `X_train` and `y_train`, then use that fitted model to predict
from `X_test`. Pass the **raw mixed-type DataFrames** to the pipeline.

**Explain:** Why should `passed` be excluded from the input features? Which step learns
the median, and which rows should it use? How does the pipeline handle a missing test value?

In [ ]:
fitted_course_model = None   # TODO: fit course_model on the training inputs and labels.

if fitted_course_model is None:
    print("Complete the fitting call and the prediction below.")
else:
    course_predictions = None   # TODO: predict labels for X_test.
    if course_predictions is None:
        print("Complete the prediction call.")
    else:
        assert len(course_predictions) == len(X_test), "Predict on the held-out rows."
        print("Test accuracy:", round(accuracy_score(y_test, course_predictions), 3))
        prepared = fitted_course_model.named_steps["preprocess"].transform(X_test)
        assert np.isfinite(prepared).all(), "Missing values should be imputed."
        print("Transformed test shape:", prepared.shape)
        print("Output columns:", fitted_course_model.named_steps["preprocess"].get_feature_names_out())

**Your explanation:** Replace this sentence with your answer.

## 4. Stratification, class imbalance and SMOTE · 15 minutes

The slides use credit-card fraud detection. We will use a small synthetic analogue with
**95% legitimate transactions and 5% fraud**, so no Kaggle download is needed.
The two numeric signals are artificial, not real transaction fields.

Stratification preserves the approximate class ratio in a split. Resampling changes the
**training** class ratio. They serve different purposes.

**Before running:** What accuracy could a model achieve by always predicting legitimate?

In [ ]:
fraud_X, fraud_y = make_classification(
    n_samples=1000, n_features=2, n_informative=2, n_redundant=0,
    n_clusters_per_class=1, weights=[0.95, 0.05], flip_y=0,
    class_sep=0.9, random_state=SEED,
)
# Deliberately different units: scale before distance-based SMOTE.
fraud_X = fraud_X * np.array([1000.0, 1.0])
F_train, F_test, f_train, f_test = train_test_split(
    fraud_X, fraud_y, test_size=0.25, stratify=fraud_y, random_state=SEED
)

def class_counts(labels):
    return pd.Series(labels).value_counts().reindex([0, 1], fill_value=0).rename(
        index={0: "Legitimate (0)", 1: "Fraud (1)"}
    )

display(pd.DataFrame({"All rows": class_counts(fraud_y),
                      "Training": class_counts(f_train),
                      "Test": class_counts(f_test)}))

def metric_row(name, truth, prediction):
    return {"Model": name, "Accuracy": accuracy_score(truth, prediction),
            "Balanced accuracy": balanced_accuracy_score(truth, prediction),
            "Fraud precision": precision_score(truth, prediction, zero_division=0),
            "Fraud recall": recall_score(truth, prediction, zero_division=0),
            "Fraud F1": f1_score(truth, prediction, zero_division=0)}

majority_model = DummyClassifier(strategy="most_frequent").fit(F_train, f_train)
majority_prediction = majority_model.predict(F_test)
display(pd.DataFrame([metric_row("Always legitimate", f_test, majority_prediction)]).round(3))

Fraud **recall** is the fraction of frauds caught. Fraud **precision** is the fraction of
fraud alerts that are correct. **F1** combines precision and recall. **Balanced accuracy**
is the mean of the two class recalls. Always predicting legitimate gives zero fraud recall.
When there are no positive predictions, this example reports precision as 0 (`zero_division=0`).

### Worked example: random under- and oversampling

Undersampling removes majority-class rows. Random oversampling repeats minority-class rows.
SMOTE creates new minority-class points between neighbouring minority-class points.
All these operations belong on training data only.

In [ ]:
# This fitted scaler and the resampled arrays are for inspecting the training data.
# The evaluation pipelines below will fit their own scalers on raw training inputs.
fraud_scaler = StandardScaler()
F_train_scaled = fraud_scaler.fit_transform(F_train)
resampling_counts = {"Original train": class_counts(f_train)}
for name, sampler in {
    "Random under": RandomUnderSampler(random_state=SEED),
    "Random over": RandomOverSampler(random_state=SEED),
}.items():
    sampled_X, sampled_y = sampler.fit_resample(F_train_scaled, f_train)
    resampling_counts[name] = class_counts(sampled_y)
display(pd.DataFrame(resampling_counts))
print("The test partition still has", len(f_test), "original observations.")

### How SMOTE creates a point

For a minority point $x_i$, a nearby minority point $x_j$, and $0\leq\lambda\leq1$:

$$x_{new}=x_i+\lambda(x_j-x_i).$$

The slide example uses $x_i=(2,2)$, $x_j=(4,4)$ and $\lambda=0.25$.
Predict the result, then run the cell.

In [ ]:
point_a = np.array([2.0, 2.0])
point_b = np.array([4.0, 4.0])
new_point = point_a + 0.25 * (point_b - point_a)
print("Synthetic point:", new_point)

### Exercise 4 · Generate synthetic training observations


Create `SMOTE(random_state=SEED, k_neighbors=5)` and apply `fit_resample` to
`F_train_scaled` and `f_train`. Replace the two `None` values. This exercise shows what
resampling does; the following worked example shows how to evaluate it in a pipeline.

**Explain:** Which class changes size? How does SMOTE differ from random oversampling?
Why must the test data remain untouched?

In [ ]:
smote_sampler = None    # TODO: construct SMOTE with the supplied settings.

if smote_sampler is None:
    print("Create the SMOTE sampler and complete resampling below.")
else:
    smote_result = None    # TODO: fit_resample using only scaled training inputs and training labels.
    if smote_result is None:
        print("Complete the resampling call.")
    else:
        F_smote, f_smote = smote_result
        display(pd.DataFrame({"Before": class_counts(f_train), "After SMOTE": class_counts(f_smote)}))
        assert class_counts(f_smote).nunique() == 1, "Default SMOTE should balance these two classes."
        assert len(F_smote) == len(f_smote), "Inputs and labels must stay aligned."

**Your explanation:** Replace this sentence with your answer.

### Worked example: evaluate fixed alternatives fairly

Use `imblearn.pipeline.Pipeline` when a pipeline contains a sampler. The order is:
scaling, resampling, classification. Resampling runs during fitting; prediction transforms
and classifies the original held-out rows without resampling them.

Run the supplied comparison. We use the same held-out rows and classifier for each
predeclared method. This is a teaching comparison, not a hyperparameter search. In a project,
choose methods using validation or cross-validation on training data, then evaluate the
chosen approach once on a reserved test set.

In [ ]:
model_samplers = {
    "No resampling": "passthrough",
    "Random under": RandomUnderSampler(random_state=SEED),
    "Random over": RandomOverSampler(random_state=SEED),
    "SMOTE": SMOTE(random_state=SEED, k_neighbors=5),
}
results = [metric_row("Always legitimate", f_test, majority_prediction)]
fraud_models, fraud_predictions = {}, {}
for name, sampler in model_samplers.items():
    model = ImbPipeline([
        ("scale", StandardScaler()),
        ("sample", sampler),
        ("classify", LogisticRegression(max_iter=1000)),
    ])
    model.fit(F_train, f_train)  # Raw training data; scaling is inside the pipeline.
    prediction = model.predict(F_test)
    fraud_models[name], fraud_predictions[name] = model, prediction
    results.append(metric_row(name, f_test, prediction))
results_table = pd.DataFrame(results).set_index("Model")
display(results_table.round(3))

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
for ax, name in zip(axes, ["No resampling", "SMOTE"]):
    ConfusionMatrixDisplay.from_predictions(
        f_test, fraud_predictions[name], labels=[0, 1],
        display_labels=["Legitimate", "Fraud"], cmap="Blues", colorbar=False, ax=ax,
    )
    ax.set_title(name)
    ax.grid(False)
plt.tight_layout()
plt.show()

**Discuss:** Does SMOTE catch more fraud in this example? Does it generate more false
alerts? Compare recall, precision and the confusion matrices. A resampling method is not
guaranteed to improve every metric or every dataset.

This example has only continuous numeric signals. Plain SMOTE can create invalid fractional
category indicators if applied to one-hot columns. For mixed numeric and categorical data,
consider `SMOTENC`, random oversampling or class weighting with suitable preprocessing.

## 5. Exit questions · 5 minutes

Answer each in one or two sentences. Use your own words.

1. You find `scaler.fit_transform(X)` before `train_test_split`. What should change, and why?
2. A model has 95% accuracy and 0% fraud recall. Would you call it effective at finding fraud?
3. Where should SMOTE run when using cross-validation, and what happens to each validation fold?

**Before leaving:** Save your notebook with the completed code and written explanations.
You should have a flagged observation, a correctly transformed held-out value, pipeline
predictions and the class counts after SMOTE.

**My exit answers**

1. ...
2. ...
3. ...

---
# Optional extensions · outside the 60-minute session

Complete the core practical first. The extensions are runnable worked examples followed
by short questions. They do not depend on completing any student TODO cells.

## A. Four sampling strategies · approximately 5 minutes

Match the four approaches to slides 49–50. The population has 20 invented students:
12 on course A, 8 on course B, organised into four seminar groups of five.
A stratum is a category sampled from throughout the population; a cluster is a whole
natural group selected together. Here each seminar contains members from both courses.

In [ ]:
sampling_rng = np.random.default_rng(SEED)
population = pd.DataFrame({
    "id": np.arange(1, 21),
    "course": ["A"] * 12 + ["B"] * 8,
    "seminar": np.tile(["S1", "S2", "S3", "S4"], 5),
})
simple = population.sample(n=5, random_state=SEED)
start = int(sampling_rng.integers(0, 4))
systematic = population.iloc[start::4]  # Random start, then every fourth row.
_, stratified = train_test_split(population, test_size=5,
                                stratify=population["course"], random_state=SEED)
chosen_group = sampling_rng.choice(population["seminar"].unique())
cluster = population[population["seminar"] == chosen_group]
samples = {"Simple random": simple, "Systematic": systematic,
           "Stratified": stratified, "Cluster": cluster}
display(pd.DataFrame([
    {"Method": name, "Selected IDs": sorted(sample["id"].tolist()),
     "Course A": int((sample["course"] == "A").sum()),
     "Course B": int((sample["course"] == "B").sum())}
    for name, sample in samples.items()
]))

**Question:** Which method explicitly preserves the course proportions? Why might a
systematic sample be misleading if the row order has a repeating pattern?

## B. Dimensionality reduction with PCA · approximately 5 minutes

PCA is introduced on slides 3–4. It replaces input features with components that explain
variation. It changes the number of columns; it does not remove rows.
Fit imputation, scaling and PCA on training data. Use the fitted steps for held-out data.
The 95% target concerns input variance, not 95% prediction accuracy.

In [ ]:
from sklearn.decomposition import PCA

pca_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("pca", PCA(n_components=0.95, svd_solver="full")),
])
P_train = pca_pipeline.fit_transform(X_train[numeric_columns])
P_test = pca_pipeline.transform(X_test[numeric_columns])
ratios = pca_pipeline.named_steps["pca"].explained_variance_ratio_
print("Training shape before/after:", X_train[numeric_columns].shape, P_train.shape)
print("Held-out shape:", P_test.shape)
display(pd.DataFrame({"Component": np.arange(1, len(ratios) + 1),
                      "Explained variance": ratios,
                      "Cumulative variance": ratios.cumsum()}).round(3))

**Question:** Why might PCA discard a feature direction that helps predict `passed`?

## C. Numerosity reduction with cluster centroids · approximately 5 minutes

Slide 68 uses `ClusterCentroids`. The method can compress the majority class into
representative cluster centres. A centre is generally not an original observation.
This changes the number of rows, while PCA changes the number of feature columns.

In [ ]:
from imblearn.under_sampling import ClusterCentroids

centroid_sampler = ClusterCentroids(random_state=SEED, voting="soft")
F_centres, f_centres = centroid_sampler.fit_resample(F_train_scaled, f_train)
display(pd.DataFrame({"Original training": class_counts(f_train),
                      "After centroids": class_counts(f_centres)}))
print("Shapes:", F_train_scaled.shape, "becomes", F_centres.shape)

**Question:** What useful information might be lost when many legitimate transactions
are replaced by a small number of centres?

## D. Class weighting and cross-validation · approximately 5–10 minutes

Class weighting changes the training loss without adding or removing observations.
Compare it with SMOTE using cross-validation **on the training partition only**.
Passing the entire pipeline to `cross_validate` refits its preprocessing inside each fold.

In [ ]:
weighted_model = Pipeline([
    ("scale", StandardScaler()),
    ("classify", LogisticRegression(max_iter=1000, class_weight="balanced")),
])
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
cv_rows = []
for name, candidate in {"Class weights": weighted_model,
                        "SMOTE": fraud_models["SMOTE"]}.items():
    scores = cross_validate(
        clone(candidate), F_train, f_train, cv=cv,
        scoring={"balanced_accuracy": "balanced_accuracy", "recall": "recall", "f1": "f1"},
        n_jobs=1,
    )
    cv_rows.append({"Method": name,
                    "Mean balanced accuracy": scores["test_balanced_accuracy"].mean(),
                    "Mean fraud recall": scores["test_recall"].mean(),
                    "Mean fraud F1": scores["test_f1"].mean(),
                    "F1 fold SD": scores["test_f1"].std(ddof=1)})
display(pd.DataFrame(cv_rows).round(3))

**Question:** Why would it be wrong to apply SMOTE once to all training rows before
starting cross-validation, even though you never touched the final test set?

## Reference and API links

Primary teaching source: **Week02_Lecture02_Data Pre-processing_PartB(1).pptx**, supplied
by Dr Hamid Nasiri, 69 slides. The slide links in the session route identify the sections
used. Tiny worked examples are adapted from slides 24, 37 and 59. Other data are synthetic.

- [scikit-learn: preprocessing](https://scikit-learn.org/stable/modules/preprocessing.html)
- [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)
  and [RobustScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.RobustScaler.html)
- [OneHotEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html)
  and [ColumnTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html)
- [imbalanced-learn installation](https://imbalanced-learn.org/stable/install.html)
- [SMOTE](https://imbalanced-learn.org/stable/references/generated/imblearn.over_sampling.SMOTE.html)
  and [SMOTENC](https://imbalanced-learn.org/stable/references/generated/imblearn.over_sampling.SMOTENC.html)
- [Resampling pipelines](https://imbalanced-learn.org/stable/references/generated/imblearn.pipeline.Pipeline.html)
  and [data-leakage pitfalls](https://imbalanced-learn.org/stable/common_pitfalls.html)

Notebook prepared for the Week 2 practical, 7 October 2026. The installation cell specifies
the tested package pair; documentation pages may describe later releases.